# 04 — Early warning & watchlist

**Plain English:** Monitoring is forward-looking, so we flag loans that are
*deteriorating right now*: at each loan's latest observed month, is it in Stage 2
(or worse) and/or did its delinquency get worse versus the prior month? Those
loans form a **watchlist** — the table a credit officer would actually work.

**One-line terms**
- **Deteriorating** — this month's bucket is worse than last month's.
- **Watchlist** — currently-active loans in Stage 2/3 or freshly deteriorating.
- Loan IDs are **masked** in the committed snapshot (no loan-level redistribution).

We only look at loans still active at the end of the sample (not already prepaid/defaulted-out).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Take each loan's latest observed month ---------------------------------
order = {"Current": 0, "30": 1, "60": 2, "90+": 3, "Default": 4}
panel["bucket_rank"] = panel["bucket"].map(order)
panel = panel.sort_values(["loan_seq", "mob"])
panel["prev_rank"] = panel.groupby("loan_seq")["bucket_rank"].shift(1)
latest = panel.groupby("loan_seq").tail(1).copy()

# Active = not closed out by a terminal zero-balance event
latest = latest[latest["zb_code"].isna()]
latest["deteriorating"] = latest["bucket_rank"] > latest["prev_rank"]
print(f"{len(latest):,} active loans at latest observation")

14,517 active loans at latest observation


In [3]:
# Watchlist: Stage 2/3 now, or deteriorating this month ------------------
watch = latest[(latest.stage.isin(["Stage 2", "Stage 3"])) | (latest.deteriorating)].copy()
watch["loan_id"] = m.mask_loan_id(watch["loan_seq"])

# Attach the early-warning TRIGGER reason — the criterion that put each loan on the
# list (APS 220 para 33; APG 220 para 63). This is what turns a flat list into a
# workflow: each trigger maps to an action/owner/timeframe (config: watchlist_workflow).
def _trigger(r):
    if r["stage"] == "Stage 3":
        return "Stage 3 - default / credit-impaired"
    if r["deteriorating"] and r["stage"] == "Stage 2":
        return "SICR - entered watch (30+ DPD)"
    if r["deteriorating"]:
        return "Deteriorating vs prior month"
    return "Stage 2 - on watch"
watch["trigger"] = watch.apply(_trigger, axis=1)

watch = watch.sort_values(["bucket_rank", "cur_upb"], ascending=False)
cols = ["loan_id", "vintage", "prop_state", "period", "bucket", "stage", "trigger",
        "deteriorating", "loan_age", "cur_upb", "credit_score", "ltv"]
watch = watch[cols]
print(f"watchlist: {len(watch):,} loans  |  exposure on watch: ${watch.cur_upb.sum()/1e6:,.1f}m")
watch.head(15)

watchlist: 352 loans  |  exposure on watch: $48.4m


,loan_id,vintage,prop_state,period,bucket,stage,trigger,deteriorating,loan_age,cur_upb,credit_score,ltv
3695242,****7505,2008,NY,202509,Default,Stage 3,Stage 3 - default / credit-impaired,False,209,198614.15,716,75
3521710,****5276,2008,IL,202509,Default,Stage 3,Stage 3 - default / credit-impaired,False,202,95227.21,633,85
456252,****0440,2007,WI,202509,Default,Stage 3,Stage 3 - default / credit-impaired,False,221,78385.42,701,75
2532786,****8505,2007,NJ,202509,Default,Stage 3,Stage 3 - default / credit-impaired,False,214,45144.33,706,90
2467018,****1007,2007,NY,202509,90+,Stage 3,Stage 3 - default / credit-impaired,False,34,536582.39,703,80
6820802,****2606,2015,CA,202509,90+,Stage 3,Stage 3 - default / credit-impaired,False,124,417940.39,723,64
7996587,****0795,2015,TX,202509,90+,Stage 3,Stage 3 - default / credit-impaired,False,43,403589.33,649,90
7831932,****7120,2015,NY,202509,90+,Stage 3,Stage 3 - default / credit-impaired,False,119,399166.73,758,80
3160142,****4836,2008,MD,202509,90+,Stage 3,Stage 3 - default / credit-impaired,False,43,393380.69,687,95
8758167,****3583,2015,MD,202509,90+,Stage 3,Stage 3 - default / credit-impaired,False,48,379698.46,683,95


In [4]:
# Summary of the watchlist by vintage + stage, and save -----------------
wsumm = (watch.groupby(["vintage", "stage"])
         .agg(loans=("loan_id", "size"), exposure_upb=("cur_upb", "sum"))
         .reset_index())
wsumm["exposure_upb"] = wsumm["exposure_upb"].round(0)
# Trigger mix — how many loans each early-warning criterion raised.
tsumm = (watch.groupby("trigger").agg(loans=("loan_id", "size"),
         exposure_upb=("cur_upb", "sum")).reset_index().sort_values("loans", ascending=False))
tsumm["exposure_upb"] = tsumm["exposure_upb"].round(0)
watch.to_csv(m.OUT_TABLES / "04_watchlist.csv", index=False)
wsumm.to_csv(m.OUT_TABLES / "04_watchlist_summary.csv", index=False)
tsumm.to_csv(m.OUT_TABLES / "04_watchlist_triggers.csv", index=False)
print("saved watchlist + summary + trigger mix"); wsumm

saved watchlist + summary + trigger mix


,vintage,stage,loans,exposure_upb
0,2007,Stage 2,57,5609208.0
1,2007,Stage 3,28,3266672.0
2,2008,Stage 2,39,3687311.0
3,2008,Stage 3,17,2341081.0
4,2015,Stage 2,155,23749575.0
5,2015,Stage 3,56,9715655.0


In [5]:
# The early-warning WORKFLOW: trigger -> action -> owner -> timeframe -----
# APS 220 para 33 / APG 220 para 63 expect documented criteria and a timely
# response. The workflow stages live in config/risk_appetite.yaml (watchlist_workflow).
cfg = m.load_appetite()
wf = pd.DataFrame(cfg["watchlist_workflow"])
wf.to_csv(m.OUT_TABLES / "04_watchlist_workflow.csv", index=False)
wf

,stage,trigger,action,owner,timeframe
0,Watch,"Stage 2 (SICR) — 30+ DPD backstop, or a forwar...",Increase monitoring frequency; lender's review...,Credit Risk Analyst (2nd line),Within 30 days
1,Heightened risk,"Deteriorating vs prior month, or persistent wa...",Restructure / strengthen security / lower limi...,Head of Collections,Within 14 days
2,Default,Stage 3 — 90+ DPD or a credit event (unlikely-...,Move to NPL workout; specific provision; recov...,Workout / Collections,Immediate
